# Training English Sentiment Model

Phase 1 — Download & Understand the Dataset

Source: [IMDb Movie Reviews for Binary Sentiment Analysis (Kaggle)](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)

Goal of this notebook (this phase): **understand the data only**. No model training yet.

## 1. Dataset Loading

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 140)

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
DATA_PATH = ROOT / "data" / "raw" / "english" / "MovieReviewTrainingDatabase.csv"
print("Loading:", DATA_PATH.resolve())

df = pd.read_csv(DATA_PATH)
df.head()

Loading: /home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/data/raw/english/MovieReviewTrainingDatabase.csv


,sentiment,review
0,Positive,"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and w..."
1,Positive,'The Classic War of the Worlds' by Timothy Hines is a very entertaining film that obviously goes to great effort and lengths to faithfully recreate H. G. We...
2,Negative,The film starts with a manager (Nicholas Bell) giving welcome investors (Robert Carradine) to Primal Park . A secret project mutating a primal animal using ...
3,Negative,"It must be assumed that those who praised this film ('the greatest filmed opera ever,' didn't I read somewhere?) either don't care for opera, don't care for..."
4,Positive,"Superbly trashy and wondrously unpretentious 80's exploitation, hooray! The pre-credits opening sequences somewhat give the false impression that we're deal..."


## 2. Dataset Inspection

In [2]:
print("shape:", df.shape)
print("n_samples:", df.shape[0])
print("n_columns:", df.shape[1])
print("columns:", list(df.columns))

shape: (25000, 2)
n_samples: 25000
n_columns: 2
columns: ['sentiment', 'review']


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   sentiment  25000 non-null  str  
 1   review     25000 non-null  str  
dtypes: str(2)
memory usage: 31.7 MB


In [4]:
print("text column  : review")
print("label column : sentiment")
print()
print("review length (characters):")
print(df["review"].str.len().describe())

text column  : review
label column : sentiment

review length (characters):
count    25000.000000
mean      1304.844920
std        987.458519
min         52.000000
25%        695.000000
50%        965.000000
75%       1586.000000
max      13603.000000
Name: review, dtype: float64


## 3. Missing Values

In [5]:
print(df.isnull().sum())
print()
print("rows with any null:", int(df.isnull().any(axis=1).sum()))
print("empty/whitespace text:", int((df["review"].fillna("").str.strip() == "").sum()))

sentiment    0
review       0
dtype: int64

rows with any null: 0
empty/whitespace text: 0


## 4. Duplicates

In [6]:
n_full_dups = int(df.duplicated().sum())
n_text_dups = int(df.duplicated(subset=["review"]).sum())

print("full-row duplicates :", n_full_dups)
print("duplicate texts     :", n_text_dups)

conflicting = df.groupby("review")["sentiment"].nunique()
n_conflict = int((conflicting > 1).sum())
print("same text, different labels:", n_conflict)

df[df.duplicated(keep=False)].head(10)

full-row duplicates : 96
duplicate texts     : 96
same text, different labels: 0


,sentiment,review
37,Negative,"Dumb is as dumb does, in this thoroughly uninteresting, supposed black comedy. Essentially what starts out as Chris Klein trying to maintain a low profile, ..."
326,Negative,"This guy has no idea of cinema. Okay, it seems he made a few interestig theater shows in his youth, and about two acceptable movies that had success more of..."
428,Positive,"The scenes are fast-paced. the characters are great. I love Anne-Marie Johnson's acting. I really like the ending. However, I was disappointed that this m..."
531,Positive,"Wow! So much fun! Probably a bit much for normal American kids, and really it's a stretch to call this a kid's film, this movie reminded me a quite a bit of..."
612,Negative,There is no reason to see this movie. A good plot idea is handled very badly. In the middle of the movie everything changes and from there on nothing makes ...
727,Positive,This movie surprised me. Some things were 'clicheish' and some technological elements reminded me of the movie 'Enemy of the State' starring Will Smith. But...
1053,Negative,"Awful, simply awful. It proves my theory about 'star power.' This is supposed to be great TV because the guy who directed (battlestar) Titanica is the same ..."
1083,Positive,Everyone knows about this''Zero Day'' event. What I think this movie did that Elephant did not is that they made us see how these guys were. They showed the...
1228,Positive,Bette Midler is indescribable in this concert. She gives her all every time she is on stage. Whether we are laughing at her jokes and antics or dabbing our ...
1290,Negative,"I am not so much like Love Sick as I image. Finally the film express sexual relationship of Alex, kik, Sandu their triangle love were full of intenseness, f..."


## 5. Class Distribution

In [7]:
print("unique labels:", df["sentiment"].unique().tolist())
print()
print(df["sentiment"].value_counts())
print()
print((df["sentiment"].value_counts(normalize=True) * 100).round(2).astype(str) + "%")

unique labels: ['Positive', 'Negative']

sentiment
Positive    12500
Negative    12500
Name: count, dtype: int64

sentiment
Positive    50.0%
Negative    50.0%
Name: proportion, dtype: str


In [8]:
counts = df["sentiment"].value_counts().reindex(["Positive", "Negative"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot(kind="bar", ax=axes[0], color=["#2ca02c", "#d62728"], rot=0)
axes[0].set_title("English class counts")
axes[0].set_xlabel("sentiment")
axes[0].set_ylabel("count")

counts.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", colors=["#2ca02c", "#d62728"])
axes[1].set_title("English class share")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

/tmp/ipykernel_28886/2637989547.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
print("Sample review per class:\n")
for label in ["Positive", "Negative"]:
    sample = df.loc[df["sentiment"] == label, "review"].iloc[0]
    print(f"[{label}] {sample[:400]}...\n")

Sample review per class:

[Positive] With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film whic...

[Negative] The film starts with a manager (Nicholas Bell) giving welcome investors (Robert Carradine) to Primal Park . A secret project mutating a primal animal using fossilized DNA, like'Jurassik Park', and some scientists resurrect one of nature's most fearsome predators, the Sabretooth tiger or Smilodon . Scientific ambition turns deadly, however, and when the high voltage fence is opened the creature esc...



## Checkpoint — what we know before modeling

| Question | Answer |
|---|---|
| Text column | `review` |
| Label column | `sentiment` |
| Samples | **25,000** rows × 2 columns |
| Nulls | **No** |
| Duplicates | **Yes** — 96 duplicate reviews; **0 conflicting labels** |
| Balanced? | **Yes** — 12,500 / 12,500 |
| Label shape | capitalized strings: `Positive` / `Negative` |
| Neutral? | **No** — English is **binary** |

Class mix:

- `Positive`: 12,500 (50.0%)
- `Negative`: 12,500 (50.0%)

**Implications for later phases (do not train yet):**

1. Drop the 96 duplicate reviews before training.
2. Keep the binary label space as-is.
3. Arabic uses lowercase 3-class labels (`positive` / `negative` / `neutral`). Normalize label casing in the app later, but do **not** force Arabic into binary without an explicit decision.
4. Reviews are long (median ~965 characters) compared with the Arabic customer reviews.